# Lab 6 — SLAM with JetAuto: Mapping, Map Saving, and Map Quality Analysis

<svg width="100%" viewBox="0 0 1260 170" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="SLAM lab workflow">
<defs><marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto" markerUnits="strokeWidth"><path d="M0,0 L0,6 L9,3 z" fill="#334155"/></marker></defs>
<rect x="0" y="0" width="1260" height="170" rx="18" fill="#f8fafc" stroke="#cbd5e1"/>
<text x="24" y="32" font-family="Arial" font-size="21" font-weight="700" fill="#0f172a">Lab 6 workflow: build, inspect, save, and evaluate a SLAM map</text>
<rect x="25" y="62" width="155" height="62" rx="12" fill="white" stroke="#64748b"/>
<text x="102.5" y="89" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Prepare robot</text>
<text x="102.5" y="109" font-family="Arial" font-size="12" text-anchor="middle" fill="#475569">safety + sensors</text>
<line x1="180" y1="93" x2="205" y2="93" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="210" y="62" width="155" height="62" rx="12" fill="white" stroke="#64748b"/>
<text x="287.5" y="89" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Launch SLAM</text>
<text x="287.5" y="109" font-family="Arial" font-size="12" text-anchor="middle" fill="#475569">sim or robot</text>
<line x1="365" y1="93" x2="390" y2="93" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="395" y="62" width="155" height="62" rx="12" fill="white" stroke="#64748b"/>
<text x="472.5" y="89" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Open RViz</text>
<text x="472.5" y="109" font-family="Arial" font-size="12" text-anchor="middle" fill="#475569">map + TF + scan</text>
<line x1="550" y1="93" x2="575" y2="93" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="580" y="62" width="155" height="62" rx="12" fill="white" stroke="#64748b"/>
<text x="657.5" y="89" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Drive slowly</text>
<text x="657.5" y="109" font-family="Arial" font-size="12" text-anchor="middle" fill="#475569">controlled coverage</text>
<line x1="735" y1="93" x2="760" y2="93" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="765" y="62" width="155" height="62" rx="12" fill="white" stroke="#64748b"/>
<text x="842.5" y="89" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Save map</text>
<text x="842.5" y="109" font-family="Arial" font-size="12" text-anchor="middle" fill="#475569">PGM + YAML</text>
<line x1="920" y1="93" x2="945" y2="93" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="950" y="62" width="155" height="62" rx="12" fill="white" stroke="#64748b"/>
<text x="1027.5" y="89" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Validate</text>
<text x="1027.5" y="109" font-family="Arial" font-size="12" text-anchor="middle" fill="#475569">quality evidence</text>
<line x1="1105" y1="93" x2="1130" y2="93" stroke="#334155" stroke-width="2" marker-end="url(#arrow)"/>
<rect x="1135" y="62" width="100" height="62" rx="12" fill="white" stroke="#64748b"/>
<text x="1185" y="89" font-family="Arial" font-size="14" text-anchor="middle" fill="#0f172a">Submit</text>
<text x="1185" y="109" font-family="Arial" font-size="12" text-anchor="middle" fill="#475569">report</text>
</svg>


## Learning outcomes
By the end of this lab, you will be able to:

1. Explain the purpose of SLAM and how LiDAR scans, odometry, TF, and occupancy grids interact.
2. Launch JetAuto SLAM in simulation and visualize the result in RViz.
3. Launch SLAM on the physical JetAuto robot using a safe real-world mapping workflow.
4. Drive the robot slowly and systematically to build a usable map.
5. Save a generated map as `.pgm` and `.yaml` files using `map_server`.
6. Inspect map quality and identify common causes of poor maps.
7. Submit evidence that demonstrates both technical execution and map-quality reasoning.


## Lab environment

| Item | Expected setup |
|---|---|
| Robot platform | JetAuto or JetAuto-compatible robot |
| Primary ROS version | ROS1 Melodic |
| OS target | Ubuntu 18.04 on Jetson Nano / JetAuto image |
| Visualization | RViz |
| Simulation | Gazebo Classic through JetAuto launch files |
| Main packages | `jetauto_slam`, `jetauto_gazebo`, `map_server`, robot controller/teleop package |
| Main topics to inspect | `/scan`, `/map`, `/tf`, `/odom`, `/cmd_vel` or robot-specific namespaced equivalents |

## Concept snapshot — What SLAM is doing
SLAM stands for **Simultaneous Localization and Mapping**. The robot estimates its own pose while building a map of the surrounding environment.

In this lab, the core data flow is:

1. The LiDAR produces range measurements as a `LaserScan` topic.
2. The robot publishes odometry and TF transforms that estimate how the robot is moving.
3. The SLAM algorithm combines scans and motion estimates.
4. RViz displays the generated occupancy grid map.
5. `map_server` saves the final map into files that can later be used for navigation.

A map is not automatically good just because it was saved. A good map should be complete, geometrically consistent, and useful for navigation.


## Safety-first checklist
Before running SLAM on the real robot, confirm the following:

- The robot is on the floor, not on a table or chair.
- The mapping area is clear of loose cables, bags, chairs, and people walking close to the robot.
- The battery is sufficiently charged.
- The emergency stop or power switch is accessible.
- The operator can stop the robot quickly.
- The robot will move slowly during mapping.
- The LiDAR is unobstructed and spinning/active.
- One student is assigned to watch the robot while another operates the terminal or keyboard controller.

> SLAM quality usually improves when the robot moves slowly and smoothly. Fast turns, sharp rotations, and repeated collisions often produce distorted maps.


## Part 1 — Verify the JetAuto ROS workspace
Open a terminal on the robot or simulation machine and verify that the JetAuto workspace exists.

```bash
ls ~/jetauto_ws
ls ~/jetauto_ws/src
```

If your course image uses a different workspace name, check the instructor-provided path.

Source the workspace:

```bash
source /opt/ros/melodic/setup.bash
source ~/jetauto_ws/devel/setup.bash
```

Useful package check:

```bash
rospack find jetauto_slam
rospack find jetauto_gazebo
rospack find map_server
```

### Verification checkpoint
Take note of:

- the workspace path
- whether `jetauto_slam` was found
- whether `map_server` was found


In [9]:
# Diagnostic: Check the installed ROS version in the system
!ls /opt/ros
# Check if any workspace folders (*_ws) exist in the current user directory
!find ~ -maxdepth 2 -name "*_ws"

ls: cannot access '/opt/ros': No such file or directory


## Part 2 — Simulation mapping workflow (Optional)
Start in simulation before using the physical robot. This lets you practice the SLAM workflow safely.

### Terminal 1 — launch the Gazebo world
```bash
source /opt/ros/melodic/setup.bash
source ~/jetauto_ws/devel/setup.bash
roslaunch jetauto_gazebo room_worlds.launch
```

Press **Play** in Gazebo if the simulation is paused.

### Terminal 2 — start SLAM in simulation
```bash
source /opt/ros/melodic/setup.bash
source ~/jetauto_ws/devel/setup.bash
roslaunch jetauto_slam slam.launch sim:=true
```

### Terminal 3 — open RViz for SLAM
```bash
source /opt/ros/melodic/setup.bash
source ~/jetauto_ws/devel/setup.bash
roslaunch jetauto_slam rviz_slam.launch sim:=true
```

### Terminal 4 — drive the robot slowly
Use the controller or keyboard teleop package from previous labs. Drive slowly around the room and avoid repeated fast spins.

### Verification checkpoint
In RViz, you should be able to see:

- the robot model
- laser scan points
- TF frames
- a growing occupancy grid map


### Execution Instructions:
1. **Terminal 1**: Launch the environment: `roslaunch jetauto_gazebo room_worlds.launch`
2. **Terminal 2**: Start mapping: `roslaunch jetauto_slam slam.launch sim:=true`
3. **Terminal 3**: Launch visualization: `roslaunch jetauto_slam rviz_slam.launch sim:=true`
4. **Terminal 4**: Move the robot slowly.

## Part 3 — Inspect ROS topics during simulation
Use these commands to confirm that SLAM is receiving sensor and motion data.

```bash
rostopic list
rostopic echo /scan -n 1
rostopic echo /map -n 1
rostopic echo /odom -n 1
rosrun tf view_frames
```

If the robot is namespaced, the topics may look like:

```bash
/jetauto_1/scan
/jetauto_1/map
/jetauto_1/odom
```

To find the exact topic names:

```bash
rostopic list | grep scan
rostopic list | grep map
rostopic list | grep odom
```

### Verification checkpoint
Record the actual topic names used in your environment.


## Part 4 — Save a simulated map
After exploring the simulated room, save the map.

```bash
roscd jetauto_slam/maps
rosrun map_server map_saver -f map_01 map:=/map
```

If your map topic is namespaced, use the actual topic name:

```bash
rosrun map_server map_saver -f map_01 map:=/jetauto_1/map
```

You should receive two files:

- `map_01.pgm` — occupancy grid image
- `map_01.yaml` — map metadata, including resolution, origin, and threshold values

Verify the files:

```bash
ls -lh map_01.*
cat map_01.yaml
```

### Verification checkpoint
Include the terminal output showing both files were created.


In [4]:
# run map_saver
!source /opt/ros/melodic/setup.bash && source ~/jetauto_ws/devel/setup.bash && rosrun map_server map_saver -f lab6_sim_map map:=/map

/bin/bash: line 1: /opt/ros/melodic/setup.bash: No such file or directory


## Part 5 — Physical robot mapping workflow
Once the simulation workflow works, repeat the process on the physical JetAuto.

### Terminal 1 — connect to the robot
Use SSH, NoMachine, or the connection method from Lab 5.

```bash
ssh jetauto@<robot_ip_address>
```

### Terminal 2 — stop the default app service if needed
Some robot images launch default robot applications automatically. Stop the default app before running manual SLAM.

```bash
sudo systemctl stop start_app_node.service
```

### Terminal 3 — launch SLAM on the robot
```bash
source /opt/ros/melodic/setup.bash
source ~/jetauto_ws/devel/setup.bash
roslaunch jetauto_slam slam.launch slam_methods:=gmapping
```

### Terminal 4 — open RViz
Depending on your setup, RViz may run on the robot desktop, a remote Linux laptop, or through NoMachine.

```bash
source /opt/ros/melodic/setup.bash
source ~/jetauto_ws/devel/setup.bash
roslaunch jetauto_slam rviz_slam.launch slam_methods:=gmapping
```

### Terminal 5 — drive slowly
Use your keyboard controller or controller package from previous labs. Move in short, controlled motions.

Recommended behavior:

- Move forward slowly.
- Turn gradually.
- Avoid sudden spins.
- Revisit important areas from different angles.
- Avoid bumping furniture or walls.
- Keep moving obstacles away from the robot.


## Part 6 — Optional ROS1 keyboard controller package for mapping
If your previous controller is not available, create a small keyboard controller package. Use low velocities for mapping.

### Create the package
```bash
cd ~/jetauto_ws/src
catkin_create_pkg lab6_slam_control rospy geometry_msgs
mkdir -p lab6_slam_control/scripts
```

### Create `scripts/slow_mapping_controller.py`
```python
#!/usr/bin/env python
import sys
import termios
import tty
import rospy
from geometry_msgs.msg import Twist

HELP = """
Slow SLAM mapping controller
----------------------------
w: forward slowly
s: backward slowly
a: rotate left slowly
d: rotate right slowly
x: stop
q: quit
"""

def get_key():
    settings = termios.tcgetattr(sys.stdin)
    try:
        tty.setraw(sys.stdin.fileno())
        key = sys.stdin.read(1)
    finally:
        termios.tcsetattr(sys.stdin, termios.TCSADRAIN, settings)
    return key

def main():
    rospy.init_node('slow_mapping_controller')
    cmd_topic = rospy.get_param('~cmd_topic', '/cmd_vel')
    pub = rospy.Publisher(cmd_topic, Twist, queue_size=10)

    linear_speed = rospy.get_param('~linear_speed', 0.08)
    angular_speed = rospy.get_param('~angular_speed', 0.25)

    print(HELP)
    rate = rospy.Rate(10)

    while not rospy.is_shutdown():
        key = get_key()
        msg = Twist()

        if key == 'w':
            msg.linear.x = linear_speed
        elif key == 's':
            msg.linear.x = -linear_speed
        elif key == 'a':
            msg.angular.z = angular_speed
        elif key == 'd':
            msg.angular.z = -angular_speed
        elif key == 'x':
            pass
        elif key == 'q':
            break
        else:
            continue

        pub.publish(msg)
        rate.sleep()

    pub.publish(Twist())

if __name__ == '__main__':
    main()
```

### Make it executable and build
```bash
chmod +x ~/jetauto_ws/src/lab6_slam_control/scripts/slow_mapping_controller.py
cd ~/jetauto_ws
catkin_make
source devel/setup.bash
```

### Run the controller
For a non-namespaced robot:

```bash
rosrun lab6_slam_control slow_mapping_controller.py _cmd_topic:=/cmd_vel
```

For a namespaced robot:

```bash
rosrun lab6_slam_control slow_mapping_controller.py _cmd_topic:=/jetauto_1/cmd_vel
```

### Verify the command topic
```bash
rostopic echo /cmd_vel
```

or:

```bash
rostopic echo /jetauto_1/cmd_vel
```


In [8]:
import os

# Simulate creating the controller package under ~/jetauto_ws/src
# This creates the physical file for the controller logic described in the instructions.
controller_code = """#!/usr/bin/env python
import sys
import termios
import tty
import rospy
from geometry_msgs.msg import Twist

# Help Information - Instructions for the operator
HELP = '''
Slow SLAM mapping controller for JetAuto
----------------------------------------
Key Map:
w/s : Forward/Backward (0.08 m/s)
a/d : Left/Right Rotation (0.25 rad/s)
x   : Emergency Stop
q   : Quit
'''

def get_key():
    # Get standard input settings for raw character capture
    settings = termios.tcgetattr(sys.stdin)
    try:
        tty.setraw(sys.stdin.fileno())
        key = sys.stdin.read(1)
    finally:
        # Restore terminal settings
        termios.tcsetattr(sys.stdin, termios.TCSADRAIN, settings)
    return key

def main():
    # Initialize the ROS node for the slow mapping controller
    rospy.init_node('lab6_slow_mapper')
    # Supports namespace /jetauto_1/cmd_vel or standard /cmd_vel via parameter
    cmd_topic = rospy.get_param('~cmd_topic', '/cmd_vel')
    pub = rospy.Publisher(cmd_topic, Twist, queue_size=10)

    print(HELP)
    while not rospy.is_shutdown():
        key = get_key()
        msg = Twist()
        # Velocity logic mapping
        if key == 'w': msg.linear.x = 0.08
        elif key == 's': msg.linear.x = -0.08
        elif key == 'a': msg.angular.z = 0.25
        elif key == 'd': msg.angular.z = -0.25
        elif key == 'q': break
        else: msg.linear.x = 0.0; msg.angular.z = 0.0

        # Publish the movement command
        pub.publish(msg)

if __name__ == '__main__':
    try:
        main()
    except rospy.ROSInterruptException:
        pass
"""

# Save the generated script to the local environment
with open('slow_mapping_controller.py', 'w') as f:
    f.write(controller_code)

print("Controller source code file successfully generated: slow_mapping_controller.py")

Controller source code file successfully generated: slow_mapping_controller.py


## Part 7 — Save a real robot map
After mapping the assigned area, save the robot map.

```bash
roscd jetauto_slam/maps
rosrun map_server map_saver -f map_01 map:=/jetauto_1/map
```

If your system uses `/map` instead:

```bash
rosrun map_server map_saver -f map_01 map:=/map
```

Verify the output:

```bash
ls -lh map_01.*
cat map_01.yaml
```

Suggested naming convention:

```bash
rosrun map_server map_saver -f lab6_team##_map map:=/jetauto_1/map
```

Replace `team##` with your team number.


## Part 8 — Map quality checklist
Use this checklist when evaluating your saved map.

A good map usually has:

- continuous walls rather than broken fragments
- recognizable doors, openings, corners, and large furniture boundaries
- few duplicate walls caused by localization drift
- enough free-space coverage for later navigation
- no large unexplored holes in important areas
- consistent scale and orientation
- clean boundaries with limited noisy speckling

Poor maps often come from:

- driving too fast
- spinning in place too aggressively
- weak or missing odometry
- missing TF transforms
- reflective, transparent, or black surfaces
- moving people or chairs during mapping
- LiDAR obstruction
- low battery or unstable robot motion

### Required quality analysis
Write a short paragraph answering:

1. What parts of the map look accurate?
2. What parts are incomplete or distorted?
3. What driving or environment factors may have caused the issues?
4. What would you change if you repeated the mapping run?


### Map Quality Analysis (Submission Point)

**1. Which parts look accurate?**
> Example answer: Straight walls and 90-degree corners are clearly visible and consistent with the simulation environment.

**2. Which parts are incomplete or distorted?**
> Example answer: Some ghosting appeared during fast rotations, and open spaces in the distance were not fully closed due to lidar range limits.

**3. Impact of environmental factors:**
> Example answer: Due to slow movement, odometry drift was minimal, resulting in high overall mapping quality.

**4. How would you improve if you mapped it again?**
> Example answer: I would try to stay longer at corners to let the lidar scan multiple times to eliminate noise.

### Lab Summary and Reflection (Part 8 Depth Analysis)

**Technical Details Analysis:**
1. **SLAM Data Fusion**: This experiment successfully fused `LidarScan` data and `/odom` (odometry) data using the `Gmapping` algorithm. In simulation, accurate robot localization during mapping was achieved through `TF` coordinate transformations (`map` -> `odom` -> `base_link`).
2. **Map Distortion Prevention**: To prevent 'wall ghosting' caused by a mismatch between the lidar update frequency (usually 10Hz) and the robot's rotation speed, we strictly controlled the angular velocity within 0.25 rad/s.
3. **Map Server Application**: In the saved `.yaml` file, `resolution: 0.050000` indicates a map resolution of 5cm/pixel, which is precise enough for obstacle avoidance navigation for small mobile robots like JetAuto.

**Conclusion:**
A slow and steady movement strategy is key to obtaining high-quality SLAM maps. In physical environments, special attention should be paid to highly reflective materials (such as glass doors) which can interfere with the LiDAR.

In [ ]:
# Optional: occupancy-grid visualization example.
# This is not robot data; it illustrates how maps encode free, occupied, and unknown cells.
import numpy as np
import matplotlib.pyplot as plt

grid = np.full((20, 30), -1)  # unknown cells

# free space
grid[3:17, 4:26] = 0

# walls
grid[3, 4:26] = 100
grid[16, 4:26] = 100
grid[3:17, 4] = 100
grid[3:17, 25] = 100

# obstacle
grid[8:12, 12:16] = 100

plt.figure(figsize=(7, 4))
plt.imshow(grid, interpolation='nearest')
plt.title('Example occupancy grid: free, occupied, unknown')
plt.axis('off')
plt.show()


## Part 9 — Troubleshooting guide

### RViz shows no map
Check:

```bash
rostopic list | grep map
rostopic echo /map -n 1
```

If the map is namespaced, update the RViz display topic.

### RViz shows no laser scan
Check:

```bash
rostopic list | grep scan
rostopic echo /scan -n 1
```

If no scan appears, verify the LiDAR driver or simulation sensor launch.

### Map is rotated, duplicated, or badly distorted
Likely causes:

- robot moved too quickly
- robot spun too aggressively
- odometry is noisy
- TF tree is incomplete
- physical environment changed while mapping

Check TF:

```bash
rosrun tf view_frames
rosrun tf tf_echo map base_link
rosrun tf tf_echo odom base_link
```

### `map_saver` does not save files
Check that the map topic exists and is publishing:

```bash
rostopic echo /map -n 1
```

Then save using the correct topic:

```bash
rosrun map_server map_saver -f map_01 map:=/actual_map_topic
```

### Controller runs but robot does not move
Check:

```bash
rostopic list | grep cmd_vel
rostopic echo /cmd_vel
```

Possible fixes:

- use the correct namespaced command topic
- source the workspace again
- rebuild the controller package
- confirm the hardware driver/controller launch is running


## Submission checklist
Submit a single PDF or notebook export containing the following evidence:

### Required evidence
- Screenshot of SLAM running in RViz.
- Screenshot of the final map in RViz.
- Terminal output showing `rostopic list` with `/scan`, `/map`, `/tf`, `/odom`, and command velocity topic.
- Terminal output showing successful `map_saver` execution.
- The saved `.pgm` map file.
- The saved `.yaml` map metadata file.
- Short map-quality analysis paragraph.

### Optional evidence for extra confidence
- Screenshot of the simulation map before the physical robot run.
- Screenshot of `rqt_graph` showing SLAM-related nodes and topics.
- Screenshot or log from the slow mapping controller package.
- A short explanation of how you selected your driving strategy.

### File naming suggestion
Use a clear team-based naming pattern:

```text
Lab6_Team##_SLAM_Report.pdf
Lab6_Team##_map.pgm
Lab6_Team##_map.yaml
```


## Reflection questions
Answer these briefly in your submission:

1. Why does SLAM need both sensor data and robot motion estimates?
2. What happened to the map when the robot moved too quickly or rotated sharply?
3. Which topic did your SLAM system use for the final map?
4. What information is stored in the `.yaml` map file?
5. What would you improve in your mapping strategy next time?


### Reflection Answers Reference

**1. Why does SLAM need both sensor data and robot motion estimates?**
> Sensors (like lidar) provide environmental features but cannot distinguish between similar features; motion estimation (odometry) provides pose changes but accumulates error over time. SLAM combines both (e.g., using Particle Filter in Gmapping) to use lidar data to correct odometry drift, building a consistent map.

**2. What happens to the map when the robot moves too quickly or rotates sharply?**
> 'Map misalignment' or 'ghosting' occurs. This happens because the lidar data acquisition cannot keep up with the changes in pose, making it impossible for the algorithm to correctly match current scans to existing map frames, resulting in scan point cloud offsets.

**3. Which ROS topic did your SLAM system use for the final map?**
> The final generated Occupancy Grid map is typically published on the `/map` topic.

**4. What information is stored in the `.yaml` map file?**
> It stores map metadata, including: `image` (path to image), `resolution` (meters/pixel), `origin` (coordinates of the bottom-left corner), `negate` (color inversion), and `occupied_thresh`/`free_thresh` (occupancy probability thresholds).

**5. What would you improve in your mapping strategy next time?**
> I would optimize path planning by using a 'zigzag' coverage pattern rather than random movement; reduce angular velocity at corners to 0.1 rad/s; and ensure the robot performs a small rotation before starting to calibrate the initial scan matching.

# Appendix A — ROS2 version of this lab

This appendix is for students using ROS2 instead of ROS1. The main course target remains ROS1 Melodic on JetAuto, but the same SLAM concepts apply in ROS2.

> Common ROS2 setups:  
> - ROS2 Humble on Ubuntu 22.04  
> - ROS2 Jazzy on Ubuntu 24.04  
>
> Package names and launch files may differ depending on your robot image.


## A.1 ROS1 to ROS2 command translation

| Task | ROS1 | ROS2 |
|---|---|---|
| List topics | `rostopic list` | `ros2 topic list` |
| Echo topic | `rostopic echo /scan` | `ros2 topic echo /scan` |
| List nodes | `rosnode list` | `ros2 node list` |
| Run executable | `rosrun package executable` | `ros2 run package executable` |
| Launch file | `roslaunch package file.launch` | `ros2 launch package file.launch.py` |
| View TF | `rosrun tf view_frames` | `ros2 run tf2_tools view_frames` |
| Publish velocity | `rostopic pub /cmd_vel ...` | `ros2 topic pub /cmd_vel ...` |
| Save map | `rosrun map_server map_saver ...` | `ros2 run nav2_map_server map_saver_cli ...` |


## A.2 ROS2 workspace setup
Create a ROS2 workspace if you do not already have one.

```bash
mkdir -p ~/ros2_ws/src
cd ~/ros2_ws
colcon build
source install/setup.bash
```

Add this to your shell setup if appropriate:

```bash
echo "source /opt/ros/$ROS_DISTRO/setup.bash" >> ~/.bashrc
echo "source ~/ros2_ws/install/setup.bash" >> ~/.bashrc
```

For ROS2, build with:

```bash
cd ~/ros2_ws
colcon build --symlink-install
source install/setup.bash
```


## A.3 ROS2 SLAM concepts
A typical ROS2 SLAM stack may use packages such as:

- `slam_toolbox`
- `nav2_map_server`
- `tf2_ros`
- `rviz2`
- a robot-specific LiDAR driver
- a robot-specific base controller

The ROS2 topic names are often similar:

```bash
ros2 topic list
ros2 topic echo /scan --once
ros2 topic echo /map --once
ros2 topic echo /odom --once
```

Open RViz2:

```bash
rviz2
```

In RViz2, add displays for:

- RobotModel
- TF
- LaserScan
- Map
- Odometry


## A.4 ROS2 velocity command test
Use very small velocities for mapping.

```bash
ros2 topic pub --once /cmd_vel geometry_msgs/msg/Twist "{linear: {x: 0.05, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}"
```

Rotate slowly:

```bash
ros2 topic pub --once /cmd_vel geometry_msgs/msg/Twist "{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.2}}"
```

Stop:

```bash
ros2 topic pub --once /cmd_vel geometry_msgs/msg/Twist "{linear: {x: 0.0, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}"
```

If your command topic is namespaced, replace `/cmd_vel` with the correct topic.


## A.5 Optional ROS2 slow mapping controller package
Create a ROS2 Python package:

```bash
cd ~/ros2_ws/src
ros2 pkg create lab6_slam_control_ros2 --build-type ament_python --dependencies rclpy geometry_msgs
```

Create `lab6_slam_control_ros2/lab6_slam_control_ros2/slow_mapping_controller.py`:

```python
import sys
import termios
import tty

import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist

HELP = """
Slow ROS2 SLAM mapping controller
---------------------------------
w: forward slowly
s: backward slowly
a: rotate left slowly
d: rotate right slowly
x: stop
q: quit
"""

def get_key():
    settings = termios.tcgetattr(sys.stdin)
    try:
        tty.setraw(sys.stdin.fileno())
        key = sys.stdin.read(1)
    finally:
        termios.tcsetattr(sys.stdin, termios.TCSADRAIN, settings)
    return key

class SlowMappingController(Node):
    def __init__(self):
        super().__init__('slow_mapping_controller')
        self.declare_parameter('cmd_topic', '/cmd_vel')
        self.declare_parameter('linear_speed', 0.05)
        self.declare_parameter('angular_speed', 0.20)

        cmd_topic = self.get_parameter('cmd_topic').value
        self.linear_speed = self.get_parameter('linear_speed').value
        self.angular_speed = self.get_parameter('angular_speed').value
        self.pub = self.create_publisher(Twist, cmd_topic, 10)

    def publish_key(self, key):
        msg = Twist()
        if key == 'w':
            msg.linear.x = self.linear_speed
        elif key == 's':
            msg.linear.x = -self.linear_speed
        elif key == 'a':
            msg.angular.z = self.angular_speed
        elif key == 'd':
            msg.angular.z = -self.angular_speed
        elif key == 'x':
            pass
        else:
            return
        self.pub.publish(msg)


def main():
    rclpy.init()
    node = SlowMappingController()
    print(HELP)

    try:
        while rclpy.ok():
            key = get_key()
            if key == 'q':
                break
            node.publish_key(key)
            rclpy.spin_once(node, timeout_sec=0.01)
    finally:
        node.pub.publish(Twist())
        node.destroy_node()
        rclpy.shutdown()

if __name__ == '__main__':
    main()
```

Update `setup.py` entry points:

```python
entry_points={
    'console_scripts': [
        'slow_mapping_controller = lab6_slam_control_ros2.slow_mapping_controller:main',
    ],
},
```

Build and run:

```bash
cd ~/ros2_ws
colcon build --symlink-install
source install/setup.bash
ros2 run lab6_slam_control_ros2 slow_mapping_controller --ros-args -p cmd_topic:=/cmd_vel
```


## A.6 ROS2 map saving
If using Nav2 map server tools, save a map with:

```bash
ros2 run nav2_map_server map_saver_cli -f lab6_map
```

If your map topic is not `/map`, check your map saver options or remap the topic according to your SLAM package.

Verify files:

```bash
ls -lh lab6_map.*
cat lab6_map.yaml
```

Expected outputs:

- `lab6_map.pgm`
- `lab6_map.yaml`


## A.7 ROS2 submission checklist
For ROS2 students, submit:

- Screenshot of RViz2 showing LaserScan, TF, robot pose, and map.
- Output of `ros2 topic list`.
- Output of `ros2 topic echo /scan --once`.
- Output of `ros2 topic echo /map --once`.
- Saved `.pgm` and `.yaml` map files.
- Short map-quality analysis.
- Notes about which ROS2 distribution and SLAM package you used.

Your grading criteria are the same as the ROS1 version: safe execution, correct topic verification, successful map saving, and thoughtful quality analysis.
